# Vertex AI RAG

## Imports

In [1]:
import vertexai
from google.cloud import aiplatform
from kfp import dsl, compiler
from kfp.registry import RegistryClient
from kfp.dsl import component, Input, Output, Dataset, Artifact
from google.cloud.aiplatform import pipeline_jobs

PROJECT = "moodels-educ"
LOCATION = "us-central1"
BUCKET = "gs://experiment-b77ffdf0-1101-44eb-8d3c-3375c185554d"
PIPELINE_ROOT = f"{BUCKET}/pipeline_root"

vertexai.init(project=PROJECT, location=LOCATION)
aiplatform.init(project=PROJECT, location=LOCATION)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

## Preparation

Before executing pipelines we have to create an index where everything can be stored to.

In [2]:
INDEX = "6765631496263827456"
INDEX_RESOURCE_NAME = "projects/279637354150/locations/us-central1/indexes/" + INDEX
ENDPOINT_RESOURCE_NAME = "projects/279637354150/locations/us-central1/indexEndpoints/2424249416408891392"

In [3]:
def prepare_index_once():
    # Run this ONCE before ever running the pipeline
    aiplatform.init(project="moodels-educ", location=LOCATION)
    
    # 1. Create the index
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name="my-rag-index",
        dimensions=768,
        approximate_neighbors_count=10,
        leaf_node_embedding_count=500,
        index_update_method="STREAM_UPDATE",  # needed for upsert_datapoints
    )
    INDEX_RESOURCE_NAME = index.resource_name
    print("Index:", INDEX_RESOURCE_NAME)
    
    # 2. Create the endpoint
    endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
        display_name="my-rag-endpoint",
        public_endpoint_enabled=True,
    )
    ENDPOINT_RESOURCE_NAME = endpoint.resource_name
    print("Endpoint:", ENDPOINT_RESOURCE_NAME)
    
    # 3. Deploy index to endpoint (~10-20 min)
    endpoint.deploy_index(
        index=index,
        deployed_index_id="my_deployed_index",
    )
    print("All done — save these IDs!")

if INDEX_RESOURCE_NAME == "" and ENDPOINT_RESOURCE_NAME == "":
    prepare_index_once()
else:
    print(f"Index already created {INDEX_RESOURCE_NAME}, {ENDPOINT_RESOURCE_NAME}")

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/279637354150/locations/us-central1/indexes/6765631496263827456/operations/7612419903949111296
MatchingEngineIndex created. Resource name: projects/279637354150/locations/us-central1/indexes/6765631496263827456
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/279637354150/locations/us-central1/indexes/6765631496263827456')
Index: projects/279637354150/locations/us-central1/indexes/6765631496263827456
Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/279637354150/locations/us-central1/indexEndpoints/2424249416408891392/operations/7842033116200828928
MatchingEngineIndexEndpoint created. Resource name: projects/279637354150/locations/us-central1/indexEndpoints/2424249416408891392
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/279637354150/lo

## Components

In [4]:
@component(base_image="python:3.10", packages_to_install=[
    "google-cloud-aiplatform", "pypdf"
])
def chunk_documents(
    gcs_uri: str,
    chunk_size: int,
    chunks_out: Output[Dataset]
):
    import json, pypdf, re
    from google.cloud import storage

    # Download from GCS
    bucket_name = gcs_uri.replace("gs://", "").split("/")[0]
    blob_path = "/".join(gcs_uri.replace("gs://", "").split("/")[1:])
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(blob_path)
    blob.download_to_filename("/experiment-files/cars.txt")

    # Chunk
    # reader = pypdf.PdfReader("/tmp/doc.pdf")
    # full_text = "\n".join(p.extract_text() for p in reader.pages)
    full_text = blob.download_as_text()
    chunks = [full_text[i:i+chunk_size] for i in range(0, len(full_text), chunk_size - 50)]

    with open(chunks_out.path, "w") as f:
        json.dump(chunks, f)


In [5]:
@component(base_image="python:3.10", packages_to_install=[
    "google-cloud-aiplatform"
])
def embed_and_index(
    chunks_in: Input[Dataset],
    index_resource_name: str,
    embeddings_out: Output[Artifact]
):
    import json
    from google.cloud import aiplatform
    from vertexai.language_models import TextEmbeddingModel

    with open(chunks_in.path) as f:
        chunks = json.load(f)

    model = TextEmbeddingModel.from_pretrained("text-embedding-005")
    embeddings = model.get_embeddings(chunks)
    vectors = [e.values for e in embeddings]

    # Upsert to Vector Search
    index = aiplatform.MatchingEngineIndex(index_resource_name)
    index.upsert_datapoints(datapoints=[
        {"datapoint_id": f"chunk_{i}", "feature_vector": vec}
        for i, vec in enumerate(vectors)
    ])

    # Save chunk text mapping to GCS artifact
    data = {f"chunk_{i}": text for i, text in enumerate(chunks)}
    with open(embeddings_out.path, "w") as f:
        json.dump(data, f)


## Pipelines

In [6]:
@dsl.pipeline(name="rag-ingest", pipeline_root=PIPELINE_ROOT)
def ingest_pipeline(
    gcs_uri: str,
    index_resource_name: str,
    chunk_size: int = 500,
):
    chunk_task = chunk_documents(
        gcs_uri=gcs_uri,
        chunk_size=chunk_size,
    )
    embed_and_index(
        chunks_in=chunk_task.outputs["chunks_out"],
        index_resource_name=index_resource_name,
    )

## Executing Everything

In [8]:
# Compile
compiler.Compiler().compile(ingest_pipeline, "ingest_pipeline.yaml")

# Run ingest
ingest_job = pipeline_jobs.PipelineJob(
    display_name="rag-ingest-run",
    template_path="ingest_pipeline.yaml",
    parameter_values={
        "gcs_uri": BUCKET,
        "index_resource_name": INDEX_RESOURCE_NAME,
    }
)
ingest_job.submit()

Creating PipelineJob
PipelineJob created. Resource name: projects/279637354150/locations/us-central1/pipelineJobs/rag-ingest-20260319163734
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/279637354150/locations/us-central1/pipelineJobs/rag-ingest-20260319163734')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/rag-ingest-20260319163734?project=279637354150
